# Final results

Every table has the layout of the MPLMM paper's main table: the **available** modalities of each test case ({a}, {v}, {t}, {a,v}, {a,t}, {v,t}), ACC and F1 for each, and their average (Avg.).

| Table | Source |
|---|---|
| Paper | MPLMM paper, "Ours" row |
| Baseline | `MPLMM/results/mplmm_baseline_results.csv` |
| RAGPrompt | `RAGPrompt/results/final_rag_table.csv` (K chosen on validation) |
| MoEPrompt | `MoEPrompt/results/runs.csv` (configuration chosen on validation) |

* ACC = accuracy and F1 = weighted F1, both in %, on the **test** split.
* MOSI / MOSEI / CH-SIMS: binary sentiment (positive = score ≥ 0, CH-SIMS: > 0). IEMOCAP: 4 emotion classes.
* **Difference = ours − paper** (positive = we are above the paper).

In [1]:
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
CASES = ["{a}", "{v}", "{t}", "{a,v}", "{a,t}", "{v,t}", "Avg."]
DATASETS = ["MOSI", "IEMOCAP", "CH-SIMS", "MOSEI"]  # paper order
COLUMNS = pd.MultiIndex.from_product([CASES, ["ACC", "F1"]])

# MPLMM paper, "Ours" row: ACC, F1 for each case in CASES order.
PAPER = {
    "MOSI":    [62.71, 63.65, 63.12, 63.74, 80.12, 80.31, 65.02, 65.41, 80.76, 81.09, 81.12, 81.19, 72.14, 72.57],
    "IEMOCAP": [59.77, 59.71, 57.61, 56.98, 69.23, 69.28, 67.26, 67.37, 75.98, 75.44, 74.68, 74.51, 67.42, 67.22],
    "CH-SIMS": [65.93, 77.10, 66.02, 78.86, 79.75, 78.74, 65.28, 74.02, 77.45, 77.84, 77.97, 77.95, 72.07, 77.42],
    "MOSEI":   [67.33, 68.71, 67.29, 69.40, 79.12, 79.17, 68.21, 69.91, 80.45, 80.43, 80.11, 80.13, 73.75, 74.68],
}
paper = pd.DataFrame.from_dict(PAPER, orient="index", columns=COLUMNS)


def from_wide(df, prefix=""):
    """Rows with columns '<prefix>{a} ACC', '<prefix>{a} F1', ... -> paper layout."""
    return pd.DataFrame(
        [[row[f"{prefix}{c} {m}"] for c in CASES for m in ("ACC", "F1")] for _, row in df.iterrows()],
        index=df.index, columns=COLUMNS,
    ).loc[DATASETS]


try:  # coloured tables need pandas' Styler (a recent jinja2); otherwise plain tables
    from pandas.io.formats.style import Styler  # noqa: F401
    STYLED = True
except (ImportError, AttributeError):
    STYLED = False


def show(table):
    return table.style.format("{:.2f}") if STYLED else table.round(2)


def show_diff(table):
    """table - paper; green = above the paper, red = below."""
    diff = table - paper.loc[table.index]
    if not STYLED:
        return diff.map(lambda v: f"{v:+.2f}")
    color = lambda v: f"color: {'#1a7f37' if v > 0 else '#cf222e' if v < 0 else 'inherit'}"
    return diff.style.format("{:+.2f}").map(color)

## Paper (MPLMM, "Ours")

The reference every difference table is compared with.

In [2]:
show(paper)

## 1. Baseline

MPLMM as released, reproduced on our machine: frozen MOSEI-pretrained backbone, prompts trained per dataset.

In [3]:
baseline = from_wide(pd.read_csv(ROOT / "MPLMM" / "results" / "mplmm_baseline_results.csv").set_index("Dataset"))
show(baseline)

In [4]:
show_diff(baseline)  # baseline - paper

## 2. RAGPrompt

The generative prompt is built per sample from the top-K most similar training samples. K ∈ {1, 5, 10}, chosen per dataset on **validation**.

In [9]:
import numpy as np

NAMES = {"mosi": "MOSI", "iemocap": "IEMOCAP", "sims": "CH-SIMS", "mosei": "MOSEI"}


def ablation(runs, variant):
    """Every run of a sweep in the paper layout, plus the validation score used to choose
    one per dataset (✓ = highest validation Avg. ACC). Test values are shown for every run."""
    runs = runs.assign(Dataset=pd.Categorical(runs["dataset"].map(NAMES), DATASETS, ordered=True))
    runs = runs.sort_values(["Dataset", variant])
    table = pd.DataFrame(
        [[r[f"test {c} {m}"] for c in CASES for m in ("ACC", "F1")] for _, r in runs.iterrows()],
        index=pd.MultiIndex.from_arrays([runs["Dataset"], runs[variant]], names=["Dataset", variant]),
        columns=COLUMNS,
    )
    best = runs.groupby("Dataset", observed=True)["valid Avg. ACC"].transform("max")
    table.insert(0, ("Selection", "valid Avg. ACC"), runs["valid Avg. ACC"].to_numpy())
    table.insert(1, ("Selection", "chosen"), np.where(runs["valid Avg. ACC"] == best, "✓", ""))
    if not STYLED:
        return table.round(2)
    numeric = [c for c in table.columns if c != ("Selection", "chosen")]
    bold = lambda row: ["font-weight: bold" if row[("Selection", "chosen")] == "✓" else ""] * len(row)
    return table.style.format("{:.2f}", subset=numeric).apply(bold, axis=1)


# RAGPrompt ablation: every K before choosing one (test values; K chosen on validation).
rag_sweep = pd.read_csv(ROOT / "RAGPrompt" / "results" / "runs.csv")
rag_sweep = rag_sweep[rag_sweep["mode"] == "rag"].assign(K=lambda d: d["k"].astype(int))
ablation(rag_sweep, "K")


In [5]:
rag_runs = pd.read_csv(ROOT / "RAGPrompt" / "results" / "final_rag_table.csv")
rag_runs = rag_runs[rag_runs["selected"] == "✓"].set_index("Dataset")
rag = from_wide(rag_runs)
print("Selected K:", rag_runs.loc[DATASETS, "K"].to_dict())
show(rag)

Selected K: {'MOSI': 10, 'IEMOCAP': 5, 'CH-SIMS': 5, 'MOSEI': 10}


In [6]:
show_diff(rag)  # RAGPrompt - paper

## 3. MoEPrompt

Mixture of K = 3 experts added to the generator. Two configurations were trained, `MoE` and `MoE + rec + kd` (extra reconstruction and distillation losses); the one with the higher **validation** Avg. ACC is reported per dataset.

In [10]:
# MoEPrompt ablation: both configurations before choosing one (test values; chosen on validation).
moe_sweep = pd.read_csv(ROOT / "MoEPrompt" / "results" / "runs.csv")
moe_sweep["Config"] = ["MoE + rec + kd" if r or k else "MoE"
                       for r, k in zip(moe_sweep["lambda_rec"], moe_sweep["lambda_kd"])]
ablation(moe_sweep, "Config")

In [7]:
NAMES = {"mosi": "MOSI", "iemocap": "IEMOCAP", "sims": "CH-SIMS", "mosei": "MOSEI"}
moe_runs = pd.read_csv(ROOT / "MoEPrompt" / "results" / "runs.csv")
moe_runs["Dataset"] = moe_runs["dataset"].map(NAMES)
moe_runs["Config"] = ["MoE + rec + kd" if r or k else "MoE" for r, k in zip(moe_runs["lambda_rec"], moe_runs["lambda_kd"])]
moe_best = moe_runs.loc[moe_runs.groupby("Dataset")["valid Avg. ACC"].idxmax()].set_index("Dataset")
moe = from_wide(moe_best, prefix="test ")
print("Selected configuration:", moe_best.loc[DATASETS, "Config"].to_dict())
show(moe)

Selected configuration: {'MOSI': 'MoE', 'IEMOCAP': 'MoE + rec + kd', 'CH-SIMS': 'MoE + rec + kd', 'MOSEI': 'MoE + rec + kd'}


In [8]:
show_diff(moe)  # MoEPrompt - paper